In [ ]:
import torch
from cheetah.particles import ParticleBeam
from cheetah.accelerator import Solenoid, Drift, Segment

In [ ]:
# initial particle coordinates: x, px, y, py, z, pz, t
x0 = torch.tensor([1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0]) * 1e-3  # 1 mm offset in x and y
x0 = x0.unsqueeze(0)  # shape (1,7)

test_beam = ParticleBeam(x0, energy=torch.tensor(5e6))

In [ ]:
# create solenoid element
solenoid = Solenoid(
    length=torch.tensor(1.0),  # 1.0 m long solenoid
)

# set focusing strength scan range
n_steps = 10
scan_range = torch.linspace(0.0, 1.0, n_steps)  # 0.0 T to 1 T

solenoid.k = scan_range


# define lattice
lattice = Segment(elements=[solenoid, Drift(length=torch.tensor(1.0))])

In [ ]:
# track beam
final_beam = lattice.track(test_beam)

In [ ]:
# plot final beam location as a function of solenoid field
import matplotlib.pyplot as plt

plt.plot(scan_range.numpy(), final_beam.x.numpy() * 1e3, label="x position")
plt.plot(scan_range.numpy(), final_beam.y.numpy() * 1e3, label="y position")

plt.xlabel("Solenoid Field (T)")
plt.ylabel("Final Beam Position (mm)")
plt.title("Final Beam Position vs Solenoid Field")
plt.legend()
plt.show()

In [ ]:
# define a function that takes in a beam location and solenoid field and returns the final beam location
# put into a form such that it can be used in Xopt


def compute_final_beam_position(inputs):
    solenoid_field = inputs["solenoid_field"]
    initial_beam_x = inputs["initial_beam_x"] * 1e-3  # initial x centroid in meters
    initial_beam_y = inputs["initial_beam_y"] * 1e-3  # initial y centroid in meters

    # set beam initial conditions
    test_beam = ParticleBeam(torch.zeros(1, 7), energy=torch.tensor(5e6))
    test_beam.particles[:, 0] = initial_beam_x
    test_beam.particles[:, 2] = initial_beam_y

    # update solenoid field
    solenoid = Solenoid(
        length=torch.tensor(1.0),
        k=torch.tensor(solenoid_field),
    )

    # define lattice
    lattice = Segment(elements=[solenoid, Drift(length=torch.tensor(1.0))])

    # track beam
    final_beam = lattice.track(test_beam)
    return {
        "final_beam_x": float(final_beam.x) * 1e3,  # x centroid position in mm
        "final_beam_y": float(final_beam.y) * 1e3,  # y centroid position in mm
    }

In [ ]:
from xopt.vocs import VOCS

# set up VOCS
meas_param = "solenoid_field"
var_names = ["initial_beam_x", "initial_beam_y", "solenoid_field"]
variables = {var_name: [-3, 3] for var_name in var_names}
variables[meas_param] = [
    0.8,
    1,
]  # overwrite bounds for solenoid parameter to cover appropriate range


# construct vocs
vocs = VOCS(
    variables=variables,
    observables=["final_beam_x", "final_beam_y"],
)

meas_dim = sorted(vocs.variable_names).index(meas_param)
print("variable_names =", vocs.variable_names)
print("meas_param =", "'" + meas_param + "'")
print("meas_dim =", meas_dim)
print("domain =\n", vocs.bounds)

In [ ]:
from xopt import Xopt
from xopt.evaluator import Evaluator
from xopt.generators.bayesian.bax_generator import BaxGenerator
from bax_algorithms.solenoid_alignment import PathwiseSolenoidAlignment
from bax_algorithms.pathwise.optimize import DifferentialEvolution
from xopt.numerical_optimizer import LBFGSOptimizer

# Prepare Algorithm
algo_kwargs = {
    "x_key": "final_beam_x",
    "y_key": "final_beam_y",
    "n_samples": 2,
    "meas_dim": meas_dim,
    "n_steps_measurement_param": 5,
    "observable_names_ordered": ["final_beam_x", "final_beam_y"],
    "optimizer": DifferentialEvolution(minimize=True, maxiter=10, verbose=False),
    "n_batch": 5,
}
algo = PathwiseSolenoidAlignment(**algo_kwargs)

numerical_optimizer = LBFGSOptimizer(n_restarts=10, max_time=1)


# construct BAX generator
generator = BaxGenerator(
    vocs=vocs,
    numerical_optimizer=numerical_optimizer,
    algorithm=algo,
)

generator.gp_constructor.use_low_noise_prior = True
# construct evaluator
evaluator = Evaluator(function=compute_final_beam_position)

# construct Xopt optimizer
X = Xopt(evaluator=evaluator, generator=generator, vocs=vocs)

In [ ]:
X.random_evaluate(3)

In [ ]:
import time

for i in range(10):
    print(i)
    start = time.time()
    X.step()
    print(time.time() - start)

In [ ]:
X.data

In [ ]:
X.generator.algorithm.results["best_inputs"]

In [ ]:
from bax_algorithms.utils import get_bax_mean_prediction, tuning_input_tensor_to_dict

mean_optimizer = DifferentialEvolution(
    minimize=True, popsize=100, maxiter=100, verbose=True
)
x_tuning = get_bax_mean_prediction(X.generator, mean_optimizer)
x_tuning_dict = tuning_input_tensor_to_dict(X.generator, x_tuning)
print(x_tuning)
print(x_tuning_dict)
reference_point = x_tuning_dict | {"solenoid_field": 0.0}
print(reference_point)

In [ ]:
X.generator.visualize_model(
    variable_names=["initial_beam_x", "solenoid_field"], reference_point=reference_point
)

In [ ]:
X.generator.visualize_model(
    variable_names=["initial_beam_y", "solenoid_field"], reference_point=reference_point
)

In [ ]:
from bax_algorithms.visualize import visualize_virtual_measurement_result

fig, ax = visualize_virtual_measurement_result(
    X.generator,
    variable_names=["initial_beam_x", "initial_beam_y"],
    reference_point=reference_point,
    n_grid=10,
    n_samples=100,
    result_keys=["objective", "misalignment_x", "misalignment_y"],
)

In [ ]:
fig, ax = visualize_virtual_measurement_result(
    X.generator,
    variable_names=["initial_beam_x"],
    reference_point=reference_point,
    n_grid=100,
    n_samples=100,
    result_keys=["objective", "misalignment_x", "misalignment_y"],
)

In [ ]:
fig, ax = visualize_virtual_measurement_result(
    X.generator,
    variable_names=["initial_beam_y"],
    reference_point=reference_point,
    n_grid=100,
    n_samples=100,
    result_keys=["objective", "misalignment_x", "misalignment_y"],
)

In [ ]:
for i in range(5):
    print(i)
    start = time.time()
    X.step()
    print(time.time() - start)

In [ ]:
x_tuning = get_bax_mean_prediction(X.generator, mean_optimizer)
x_tuning_dict = tuning_input_tensor_to_dict(X.generator, x_tuning)
print(x_tuning)
print(x_tuning_dict)
reference_point = x_tuning_dict | {"solenoid_field": 0.0}
print(reference_point)

In [ ]:
X.generator.visualize_model(
    variable_names=["initial_beam_x", "solenoid_field"], reference_point=reference_point
)

In [ ]:
X.generator.visualize_model(
    variable_names=["initial_beam_y", "solenoid_field"], reference_point=reference_point
)

In [ ]:
fig, ax = visualize_virtual_measurement_result(
    X.generator,
    variable_names=["initial_beam_x", "initial_beam_y"],
    reference_point=reference_point,
    n_grid=11,
    n_samples=100,
    result_keys=["objective", "misalignment_x", "misalignment_y"],
)

In [ ]:
fig, ax = visualize_virtual_measurement_result(
    X.generator,
    variable_names=["initial_beam_x"],
    reference_point=reference_point,
    n_grid=100,
    n_samples=100,
    result_keys=["objective", "misalignment_x", "misalignment_y"],
)

In [ ]:
fig, ax = visualize_virtual_measurement_result(
    X.generator,
    variable_names=["initial_beam_y"],
    reference_point=reference_point,
    n_grid=100,
    n_samples=100,
    result_keys=["objective", "misalignment_x", "misalignment_y"],
)